# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [95]:
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from tqdm.notebook import tqdm
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [96]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['timestamp'] = pd.to_datetime(X['timestamp'])

        df = pd.DataFrame({
            'uid': X['uid'],
            'labname': X['labname'],
            'numTrials': X['numTrials'],
            'hour': X['timestamp'].dt.hour,
            'weekday': X['timestamp'].dt.weekday
        })
        
        return df

    def fit_transform(self, X, y=None):
        return self.fit(X).transform(X)

In [97]:
# class MyOneHotEncoder(BaseEstimator, TransformerMixin):
#     def __init__(self, target='weekday'):
#         self.target = target

#     def fit(self, X, y=None):        
#         return self

#     def transform(self, X):
#         ohe_lab = OneHotEncoder(sparse=False)
#         ohe_uid = OneHotEncoder(sparse=False)

#         if self.target!='labname':
#             labname = pd.DataFrame(
#                 ohe_lab.fit_transform(X[['labname']]),
#                 columns = [f'{name}' for name in ohe_lab.get_feature_names(['labname'])]
#             )
#         if self.target!='uid':
#             uid = pd.DataFrame(
#                 ohe_uid.fit_transform(X[['uid']]),
#                 columns = [f'{name}' for name in ohe_uid.get_feature_names(['uid'])]
#             )

#         new_df = pd.concat([X, uid, labname], axis=1).drop(columns=['uid', 'labname', self.target])
#         target = X[self.target]
#         return new_df, target

#     def fit_transform(self, X, y=None):
#         self.fit(X)
#         return self.transform(X)

In [98]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target='weekday'):
        self.target = target

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        
        # Кодируем только те признаки, которые не являются target
        if self.target != 'labname':
            X = pd.get_dummies(X, columns=['labname'], prefix='labname')
        
        if self.target != 'uid':
            X = pd.get_dummies(X, columns=['uid'], prefix='uid')
        
        # Отделяем target
        y = X[self.target]
        X = X.drop(columns=[self.target])
        
        return X, y

    def fit_transform(self, X, y=None):
        return self.transform(X)

In [99]:
class TrainValidationTest:
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def split(self):
        X_temp, X_test, y_temp, y_test = train_test_split(
            self.X, self.y,
            test_size=0.2,
            random_state=21,
            stratify=self.y
        )
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_temp, y_temp,
            test_size=0.25,
            random_state=21,
            stratify=y_temp
        )
        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [100]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.results = []
    
    def choose(self, X_train, X_valid, y_train, y_valid):
        for model in self.grids:
            estimator_name = model.estimator.__class__.__name__
            print(f'\nEstimator: {estimator_name}')
            grid = model.fit(X_train, y_train)

            param_grid = model.param_grid
            total_iterations = len(list(ParameterGrid(param_grid))) * model.cv
            for i in tqdm(range(total_iterations)):
                time.sleep(0.01)
            
            y_pred = grid.predict(X_valid)
            valid_score = accuracy_score(y_valid, y_pred)
            best_params = grid.best_params_
            best_score = grid.best_score_
                
            self.results.append({
                'model': estimator_name,
                'params': best_params,
                'valid_score': valid_score,
                'train_score': best_score
            })

            print(f'Best params: {best_params}')
            print(f'Best training accuracy: {best_score:.3f}')
            print(f'Validation set accuracy score for best params: {valid_score:.3f}')
        
    def best_results(self):
        return pd.DataFrame(self.results)

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [101]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator

    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)

        test_accuracy = self.estimator.score(X_test, y_test)

        print(f"Accuracy of the final model is {test_accuracy:.12f}")
        return test_accuracy

    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print(f"The model was successfully saved to {path}")

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [102]:
df = pd.read_csv('../data/checker_submits.csv')

In [103]:
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder('weekday'))
])

In [104]:
data, target = preprocessing.fit_transform(df)

In [105]:
X_train, X_valid, X_test, y_train, y_valid, y_test = TrainValidationTest(data, target).split()

In [106]:
svm_params = {
    'kernel': ['linear', 'rbf',  'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None],
    'random_state': [21],
    'probability': [True]
}
tree_params = {
    'max_depth': [5, 10, 15, 20, 25, 30, None],
    'class_weight': ['balanced', None],
    'criterion': ['gini', 'entropy'],
    'random_state': [21]
}
forest_params = {
    'max_depth': [1,5,10,20,30,40,45,49],
    'n_estimators' : [5,10,50,100],
    'class_weight': ['balanced', None],
    'criterion':['entropy','gini'],
    'random_state': [21]
}
svm = SVC()
tree = DecisionTreeClassifier()
forest = RandomForestClassifier()

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=10)
gs_tree = GridSearchCV(estimator=tree, param_grid=tree_params, scoring='accuracy', cv=2, n_jobs=10)
gs_rf = GridSearchCV(estimator=forest, param_grid=forest_params, scoring='accuracy', cv=2, n_jobs=10)
grids = [gs_svm, gs_tree, gs_rf]

grid_dict = {
    0: svm.__class__.__name__,
    1: tree.__class__.__name__,
    2: forest.__class__.__name__
}

In [107]:
model_selection = ModelSelection(grids, grid_dict)

best_model_name = model_selection.choose(X_train, X_valid, y_train, y_valid)

best_results_df = model_selection.best_results()
best_results_df

best_model = gs_rf.best_estimator_

final = Finalize(best_model)

test_accuracy = final.final_score(X_train, y_train, X_test, y_test)

model_filename = f"{best_model.__class__.__name__}_{test_accuracy:.5f}.sav"
final.save_model(model_filename)


Estimator: SVC


  0%|          | 0/144 [00:00<?, ?it/s]

Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.752
Validation set accuracy score for best params: 0.855

Estimator: DecisionTreeClassifier


  0%|          | 0/56 [00:00<?, ?it/s]

Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 20, 'random_state': 21}
Best training accuracy: 0.804
Validation set accuracy score for best params: 0.861

Estimator: RandomForestClassifier


  0%|          | 0/256 [00:00<?, ?it/s]

Best params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 40, 'n_estimators': 100, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.878
Accuracy of the final model is 0.908284023669
The model was successfully saved to RandomForestClassifier_0.90828.sav
